# Практикум 15. Тематическое моделирование LDA

Сопровождает интерактивную лабораторию №15 («Тематическое моделирование»). Тема курса: латентное размещение Дирихле и семантическая группировка

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.35, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

LDA работает с **мешком слов** (счётчики, не TF-IDF).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split

cv = CountVectorizer(min_df=3)
X = cv.fit_transform(df.text); terms = np.array(cv.get_feature_names_out())
lda = LatentDirichletAllocation(n_components=4, learning_method="batch", random_state=0).fit(X)
def show_topics(model, n=8):
    for i, comp in enumerate(model.components_):
        print(f"Тема {i}:", ", ".join(terms[comp.argsort()[::-1][:n]]))
show_topics(lda)

## 1. Доли тем в документах

In [ ]:
theta = lda.transform(X)
order = df.label.argsort().values
plt.figure(figsize=(8, 5)); plt.imshow(theta[order], aspect="auto", cmap="viridis"); plt.colorbar(label="доля темы")
plt.xlabel("тема"); plt.ylabel("документы (отсортированы по настоящей теме)"); plt.grid(False); plt.title("Документ-тема (θ)"); plt.show()
print(pd.crosstab(df.label, theta.argmax(1)))

## 2. Смешанные документы: LDA показывает несколько тем

In [ ]:
mixed = ["матч команда гол игрок закон депутат выборы партия", "суп рецепт соль мясо процессор программа алгоритм данные"]
print(pd.DataFrame(lda.transform(cv.transform(mixed)).round(2), index=["спорт+политика", "кулинария+технологии"]))

## 3. Выбор числа тем

In [ ]:
Xtr, Xte = train_test_split(X, test_size=0.3, random_state=0)
ks = [2, 3, 4, 5, 6, 8]; perp = [LatentDirichletAllocation(k, learning_method="batch", random_state=0).fit(Xtr).perplexity(Xte) for k in ks]
plt.plot(ks, perp, marker="o"); plt.xlabel("число тем"); plt.ylabel("perplexity на отложенных документах"); plt.title("Меньше лучше (но темы нужно читать глазами)"); plt.show()

Для интерактивной визуализации тем в реальной работе используют `pyLDAvis`, для более продвинутых моделей `gensim` и `BERTopic`.

## Задания

1. Подберите `doc_topic_prior` (α) и `topic_word_prior` (β). Как меняется «чистота» документов и тем?
2. Создайте корпус с 6 темами (добавьте свои списки слов в `TOPICS`) и проверьте, находит ли LDA 6 тем.
3. Дайте человеческие названия найденным темам и сравните с настоящими метками.